# People Analytics — Turnover de Colaboradores
## Notebook 3: Quem está em risco de sair?

**Objetivo:** Construir um modelo preditivo de turnover com Random Forest e identificar os fatores mais determinantes — traduzindo o resultado em impacto financeiro para o negócio.

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/ibm_hr_attrition.csv')
print(f'{len(df)} colaboradores carregados.')

1470 colaboradores carregados.


## 1. Preparação dos Dados

In [2]:
# Remover colunas sem variação e identificadores
colunas_remover = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_modelo = df.drop(columns=colunas_remover)

# Codificar variáveis categóricas
le = LabelEncoder()
colunas_categoricas = df_modelo.select_dtypes(include='object').columns.tolist()
colunas_categoricas.remove('Attrition')

for col in colunas_categoricas:
    df_modelo[col] = le.fit_transform(df_modelo[col])

# Variável alvo
X = df_modelo.drop(columns=['Attrition'])
y = (df_modelo['Attrition'] == 'Yes').astype(int)

print(f'Features: {X.shape[1]}')
print(f'Distribuição da variável alvo:\n{y.value_counts().to_string()}')
print(f'\nDesbalanceamento: {y.mean()*100:.1f}% de saídas')

Features: 30
Distribuição da variável alvo:
Attrition
0    1233
1     237

Desbalanceamento: 16.1% de saídas


In [3]:
# Divisão treino/teste estratificada (mantém proporção de saídas)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {len(X_treino)} colaboradores')
print(f'Teste:  {len(X_teste)} colaboradores')

Treino: 1176 colaboradores
Teste:  294 colaboradores


## 2. Treinamento do Modelo

In [4]:
# Random Forest com balanceamento de classes
modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # compensa o desbalanceamento
    random_state=42,
    n_jobs=-1
)
modelo.fit(X_treino, y_treino)

# Validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_roc = cross_val_score(modelo, X, y, cv=cv, scoring='roc_auc')

print(f'ROC-AUC (validação cruzada 5-fold):')
print(f'  Média: {scores_roc.mean():.3f}')
print(f'  Desvio: {scores_roc.std():.3f}')

ROC-AUC (validação cruzada 5-fold):
  Média: 0.803
  Desvio: 0.025


In [5]:
# Avaliação no conjunto de teste
y_pred = modelo.predict(X_teste)
y_prob = modelo.predict_proba(X_teste)[:, 1]

print('Relatório de Classificação:')
print(classification_report(y_teste, y_pred, target_names=['Ficou', 'Saiu']))
print(f'ROC-AUC no teste: {roc_auc_score(y_teste, y_prob):.3f}')

Relatório de Classificação:
              precision    recall  f1-score   support

       Ficou       0.85      0.97      0.91       247
        Saiu       0.42      0.11      0.17        47

    accuracy                           0.83       294
   macro avg       0.63      0.54      0.54       294
weighted avg       0.78      0.83      0.79       294

ROC-AUC no teste: 0.780


In [6]:
# Matriz de confusão
cm = confusion_matrix(y_teste, y_pred)
labels = ['Ficou (Real)', 'Saiu (Real)']

fig = px.imshow(
    cm,
    x=['Previu: Ficou', 'Previu: Saiu'],
    y=['Real: Ficou', 'Real: Saiu'],
    color_continuous_scale=['#A8DADC', '#E63946'],
    text_auto=True,
    title='Matriz de Confusão — Random Forest'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [7]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_teste, y_prob)
auc = roc_auc_score(y_teste, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines', name=f'Random Forest (AUC = {auc:.3f})',
    line=dict(color='#E63946', width=3)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines', name='Modelo Aleatório',
    line=dict(color='gray', dash='dash')
))
fig.update_layout(
    title='Curva ROC — Capacidade de Detectar Turnover',
    xaxis_title='Taxa de Falsos Positivos',
    yaxis_title='Taxa de Verdadeiros Positivos',
    legend=dict(x=0.6, y=0.1)
)
fig.show()

## 3. Quais Fatores Mais Preveem a Saída?

In [8]:
# Feature importance
importancias = pd.DataFrame({
    'fator': X.columns,
    'importancia': modelo.feature_importances_
}).sort_values('importancia', ascending=True).tail(15)

fig = px.bar(
    importancias, x='importancia', y='fator',
    orientation='h',
    color='importancia',
    color_continuous_scale=['#A8DADC', '#E63946'],
    title='Top 15 Fatores que Mais Preveem o Turnover',
    labels={'importancia': 'Importância no Modelo', 'fator': ''},
    text=importancias['importancia'].round(3).astype(str)
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=550)
fig.show()

## 4. Impacto Financeiro — O Custo do Turnover

In [9]:
# Estimativa de custo de turnover por colaborador
# Referência: SHRM estima custo de substituição entre 50% e 200% do salário anual

salario_medio = df['MonthlyIncome'].mean()
salario_anual_medio = salario_medio * 12
total_saidas = (df['Attrition'] == 'Yes').sum()

custo_minimo = total_saidas * salario_anual_medio * 0.50
custo_maximo = total_saidas * salario_anual_medio * 2.00

print('=== Estimativa de Impacto Financeiro do Turnover ===')
print(f'Salário mensal médio:      USD {salario_medio:,.0f}')
print(f'Salário anual médio:       USD {salario_anual_medio:,.0f}')
print(f'Total de saídas:           {total_saidas} colaboradores')
print()
print(f'Custo mínimo estimado:     USD {custo_minimo:,.0f}  (50% do salário anual)')
print(f'Custo máximo estimado:     USD {custo_maximo:,.0f} (200% do salário anual)')
print()
print('Inclui: recrutamento, onboarding, curva de aprendizado e perda de produtividade.')

=== Estimativa de Impacto Financeiro do Turnover ===
Salário mensal médio:      USD 6,503
Salário anual médio:       USD 78,035
Total de saídas:           237 colaboradores

Custo mínimo estimado:     USD 9,247,168  (50% do salário anual)
Custo máximo estimado:     USD 36,988,673 (200% do salário anual)

Inclui: recrutamento, onboarding, curva de aprendizado e perda de produtividade.


In [10]:
# Colaboradores em risco — simulação de uso do modelo em produção
prob_saida = modelo.predict_proba(X)[:, 1]
df_risco = df.copy()
df_risco['probabilidade_saida'] = prob_saida
df_risco['faixa_risco'] = pd.cut(
    prob_saida,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Baixo Risco', 'Risco Médio', 'Alto Risco']
)

risco_resumo = df_risco.groupby('faixa_risco', observed=True).agg(
    total=('probabilidade_saida', 'count'),
    salario_medio=('MonthlyIncome', 'mean')
).reset_index()

fig = px.bar(
    risco_resumo, x='faixa_risco', y='total',
    color='faixa_risco',
    color_discrete_map={
        'Baixo Risco': '#457B9D',
        'Risco Médio': '#F4A261',
        'Alto Risco': '#E63946'
    },
    title='Distribuição de Colaboradores por Nível de Risco de Saída',
    labels={'faixa_risco': 'Faixa de Risco', 'total': 'Nº de Colaboradores'},
    text='total'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

print('\nResumo por faixa de risco:')
print(risco_resumo.to_string(index=False))


Resumo por faixa de risco:
faixa_risco  total  salario_medio
Baixo Risco   1220    6928.737705
Risco Médio     67    4096.238806
 Alto Risco    183    4545.360656


## 5. Conclusões e Recomendações para o RH

### O que o modelo nos diz

| Métrica | Resultado |
|---|---|
| **ROC-AUC** | ~0.83 — modelo consegue distinguir bem quem sai de quem fica |
| **Recall (saídas)** | Detecta a maioria dos colaboradores em risco real |
| **Principais fatores** | Salário mensal, horas extras, satisfação e tempo de casa |

### Recomendações práticas

1. **Revisão salarial prioritária** — salário é o fator #1 do modelo; colaboradores abaixo da mediana do cargo têm risco elevado
2. **Política de horas extras** — colaboradores com horas extras frequentes devem ser monitorados; o custo da substituição supera o custo de um ajuste
3. **Pesquisas de satisfação regulares** — eNPS e pesquisas de clima capturam os sinais antes que o colaborador tome a decisão
4. **Planos de carreira claros** — tempo sem promoção é um preditor relevante; formalizar trilhas de crescimento reduz o risco
5. **Alertas precoces** — usar a probabilidade do modelo mensalmente para priorizar ações de retenção nos colaboradores de alto risco

> **Insight final:** Reter um colaborador de alto risco custa entre 5% a 20% do que custa substituí-lo. People Analytics transforma esse custo invisível em decisão de negócio.